## 1. Installation des dépendances

In [2]:
!pip install -q pandas openpyxl sentence-transformers rank-bm25 faiss-cpu huggingface_hub

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 58.1 MB/s eta 0:00:00


## 2. Connexion à Google Drive et Chargement du fichier Excel Multi-Feuilles

In [30]:
from google.colab import drive
import pandas as pd
import os

# 1. Montage de Google Drive dans Google Colab
drive.mount('/content/drive')

# 2. Chemin du fichier Excel dans votre Google Drive (Modifiez si nécessaire)
EXCEL_PATH = '/content/drive/MyDrive/ORIENT_IA_Dataset_Niveaux_1_a_5.xlsx'

if not os.path.exists(EXCEL_PATH):
    print(f"⚠️ Fichier non trouvé à l'emplacement '{EXCEL_PATH}'. Vérifiez votre chemin Google Drive.")
else:
    print(f"✅ Fichier trouvé : {EXCEL_PATH}")
    xls = pd.ExcelFile(EXCEL_PATH)
    print(f"Feuilles disponibles ({len(xls.sheet_names)}) :", xls.sheet_names)

    # Chargement de toutes les feuilles sous forme de dictionnaire de DataFrames
    sheets_dict = {sheet_name: pd.read_excel(xls, sheet_name=sheet_name) for sheet_name in xls.sheet_names}
    for sheet, df in sheets_dict.items():
        print(f"  * {sheet} : {len(df)} lignes")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Fichier trouvé : /content/drive/MyDrive/ORIENT_IA_Dataset_Niveaux_1_a_5.xlsx
Feuilles disponibles (7) : ['01_Formations_Matieres', 'Débouchés pro', 'Relation(parcours, métiers, com', 'Eventuelle passerelle entre for', '00_README', 'Conditions daccès en L1', 'Compétences développées']
  * 01_Formations_Matieres : 80 lignes
  * Débouchés pro : 16 lignes
  * Relation(parcours, métiers, com : 16 lignes
  * Eventuelle passerelle entre for : 40 lignes
  * 00_README : 6 lignes
  * Conditions daccès en L1 : 16 lignes
  * Compétences développées : 16 lignes


## 3. Découpage & Structuration des Chunks RAG avec Métadonnées

In [31]:
import re

def construire_chunks_rag(sheets_dict):
    """
    Transforme chaque ligne de chaque feuille Excel en un document textuel enrichi
    contenant des métadonnées précises et une citation vérifiable.
    """
    chunks = []
    for sheet_name, df in sheets_dict.items():
        if sheet_name == "00_README":
            continue

        for index, row in df.iterrows():
            if row.dropna().empty:
                continue

            fields = []
            code_parcours = row.get("Code_Parcours", row.get("Code_Parcours_Origine", None))
            mention = row.get("Mention", row.get("Mention_Origine", None))
            niveau = row.get("Niveau", None)

            for col in df.columns:
                val = row[col]
                if pd.notna(val) and str(val).strip() != "":
                    fields.append(f"{col}: {val}")

            row_text = f"FEUILLE: {sheet_name} | " + " | ".join(fields)
            citation_ref = f"[Feuille: '{sheet_name}' | Ligne {index + 2}]"

            chunks.append({
                "id": len(chunks),
                "text": row_text,
                "citation": citation_ref,
                "sheet": sheet_name,
                "row_idx": index + 2,
                "code_parcours": str(code_parcours) if pd.notna(code_parcours) else None,
                "mention": str(mention) if pd.notna(mention) else None,
                "niveau": str(niveau) if pd.notna(niveau) else None
            })
    return chunks

chunks = construire_chunks_rag(sheets_dict)
print(f"✅ {len(chunks)} chunks créés à partir du dataset Excel multi-feuilles.")
if chunks:
    print("Exemple de chunk 0 :\n", chunks[0]['citation'], "\n", chunks[0]['text'][:180], "...")

✅ 180 chunks créés à partir du dataset Excel multi-feuilles.
Exemple de chunk 0 :
 [Feuille: '01_Formations_Matieres' | Ligne 2] 
 FEUILLE: 01_Formations_Matieres | ID_Formation: INF-IGGLIA | Mention: INFORMATIQUE ET TELECOMMUNICATION | Code_Parcours: IGGLIA | Nom_Parcours: IGGLIA | Niveau: 1 | Diplôme: Bacc + ...


## 4. Pipeline de Recherche Hybride (Vectorielle + BM25 Lexicale) & Reranking

In [32]:
from sentence_transformers import SentenceTransformer
from rank_bm25 import BM25Okapi
import faiss
import numpy as np

# 1. Tokenisation pour BM25
def tokenize_fr(text):
    text_clean = re.sub(r'[^\w\s]', ' ', text.lower())
    return [w for w in text_clean.split() if len(w) > 1]

tokenized_corpus = [tokenize_fr(c["text"]) for c in chunks]
bm25 = BM25Okapi(tokenized_corpus)

# 2. Embedding Vectoriel & Index FAISS
print("Encodage des embeddings multilingues...")
embedding_model = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")
corpus_embeddings = embedding_model.encode([c["text"] for c in chunks], show_progress_bar=True).astype("float32")

# Normalisation L2 pour Cosine Similarity dans FAISS
faiss.normalize_L2(corpus_embeddings)
dimension = corpus_embeddings.shape[1]
faiss_index = faiss.IndexFlatIP(dimension)
faiss_index.add(corpus_embeddings)

# 3. Recherche Hybride + Reranking RRF (Reciprocal Rank Fusion)
def recherche_hybride(query, top_k=5, k_rrf=60):
    # A. Recherche BM25 (Lexicale)
    token_query = tokenize_fr(query)
    bm25_scores = bm25.get_scores(token_query)
    bm25_top_indices = np.argsort(bm25_scores)[::-1][:top_k * 3]

    # B. Recherche FAISS (Vectorielle)
    q_vector = embedding_model.encode([query]).astype("float32")
    faiss.normalize_L2(q_vector)
    _, faiss_top_indices = faiss_index.search(q_vector, top_k * 3)
    faiss_top_indices = faiss_top_indices[0]

    # C. Reciprocal Rank Fusion (RRF) Reranking
    rrf_scores = {}
    for rank, idx in enumerate(bm25_top_indices):
        rrf_scores[idx] = rrf_scores.get(idx, 0.0) + (1.0 / (k_rrf + rank + 1))
    for rank, idx in enumerate(faiss_top_indices):
        rrf_scores[idx] = rrf_scores.get(idx, 0.0) + (1.0 / (k_rrf + rank + 1))

    sorted_indices = sorted(rrf_scores.keys(), key=lambda x: rrf_scores[x], reverse=True)[:top_k]

    results = []
    for idx in sorted_indices:
        results.append({
            "chunk": chunks[idx],
            "score_rrf": rrf_scores[idx]
        })
    return results

print("✅ Index Hybride BM25 + FAISS avec Reranking RRF opérationnel !")

Encodage des embeddings multilingues...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Batches:   0%|          | 0/6 [00:00<?, ?it/s]

✅ Index Hybride BM25 + FAISS avec Reranking RRF opérationnel !


## 5. Création des 4 Outils Fonctionnels (Agents)

In [33]:
import json

#Toutes les filières
def lister_toutes_les_filieres():
    """Récupère la liste complète et unique de toutes les filières/mentions du dataset."""
    df = sheets_dict['01_Formations_Matieres']

    # Récupération des mentions et codes uniques
    filieres = df[['Code_Parcours', 'Nom_Parcours', 'Mention', 'Niveau']].drop_duplicates()

    resultats = []
    for idx, row in filieres.iterrows():
        resultats.append({
            "source": f"(Source : Offre de formation - Ligne {idx+2})",
            "code": row.get('Code_Parcours'),
            "nom": row.get('Nom_Parcours'),
            "mention": row.get('Mention'),
            "niveau": row.get('Niveau')
        })
    return resultats

# Outil 1 : Rechercher une formation avec filtrage ou recherche hybride
def rechercher_formation(query=None, code_parcours=None, mention=None, niveau=None):
    """Recherche des formations en combinant filtres structurés et recherche hybride RAG."""
    df = sheets_dict['01_Formations_Matieres']
    filtered = df.copy()

    if code_parcours:
        filtered = filtered[filtered['Code_Parcours'].astype(str).str.upper() == code_parcours.upper()]
    if mention:
        filtered = filtered[filtered['Mention'].astype(str).str.upper().str.contains(mention.upper())]
    if niveau:
        filtered = filtered[filtered['Niveau'].astype(str) == str(niveau)]

    results = []
    for idx, row in filtered.iterrows():
        results.append({
            "citation": f"[Feuille: '01_Formations_Matieres' | Ligne {idx+2}]",
            "Code_Parcours": row.get('Code_Parcours'),
            "Nom_Parcours": row.get('Nom_Parcours'),
            "Niveau": row.get('Niveau'),
            "Diplôme": row.get('Diplôme'),
            "Matières": row.get('Matière'),
            "Description": row.get('Description_Parcours')
        })

    if query and len(results) == 0:
        hyb_res = recherche_hybride(query, top_k=3)
        return [{"citation": r['chunk']['citation'], "details": r['chunk']['text']} for r in hyb_res]

    return results

# Outil 2 : Vérifier les prérequis
def verifier_prerequis(code_parcours):
    """Vérifie le référentiel des prérequis et compétences associées à un parcours."""
    df_ref = sheets_dict['02_Referentiel']
    rows = df_ref[df_ref['Code_Parcours'].astype(str).str.upper() == code_parcours.upper()]

    results = []
    for idx, row in rows.iterrows():
        results.append({
            "citation": f"[Feuille: '02_Referentiel' | Ligne {idx+2}]",
            "Type": row.get('Type'),
            "Element": row.get('Element'),
            "Description": row.get('Description_Detaillee'),
            "Niveau_Importance": row.get('Niveau_Importance')
        })
    return results

# Outil 3 : Calculer le score d'adéquation profil / compétences
def calculer_score_adequation(competences_profil, code_parcours):
    """Calcule le score de correspondance (%) entre le profil étudiant et les compétences d'un parcours."""
    df_comp = sheets_dict['Compétences développées']
    row = df_comp[df_comp['Code_Parcours'].astype(str).str.upper() == code_parcours.upper()]

    if row.empty:
        return {"score": "0%", "remarque": f"Parcours {code_parcours} non trouvé."}

    idx = row.index[0]
    comps_text = str(row.iloc[0].get('Compétences développées', '')).lower()

    matches = [c for c in competences_profil if c.lower() in comps_text]
    score_pct = round((len(matches) / max(len(competences_profil), 1)) * 100, 1)

    return {
        "citation": f"[Feuille: 'Compétences développées' | Ligne {idx+2}]",
        "code_parcours": code_parcours,
        "score_adequation": f"{score_pct}%",
        "competences_validees": matches,
        "toutes_competences_formation": row.iloc[0].get('Compétences développées')
    }

# Outil 4 : Identifier les débouchés professionnels et passerelles
def identifier_debouches_et_passerelles(code_parcours):
    """Récupère les débouchés métiers et les possibilités de passerelles d'un parcours."""
    df_deb = sheets_dict['Débouchés pro']
    df_pas = sheets_dict['Eventuelle passerelle entre for']

    row_deb = df_deb[df_deb['Code_Parcours'].astype(str).str.upper() == code_parcours.upper()]
    rows_pas = df_pas[df_pas['Code_Parcours_Origine'].astype(str).str.upper() == code_parcours.upper()]

    debouches = []
    for idx, r in row_deb.iterrows():
        debouches.append({
            "citation": f"[Feuille: 'Débouchés pro' | Ligne {idx+2}]",
            "Metiers_Estimes": r.get('Metiers_Estimes'),
            "Debouches_Officiels": r.get('Debouches_Officiels_Confirmes')
        })

    passerelles = []
    for idx, r in rows_pas.iterrows():
        passerelles.append({
            "citation": f"[Feuille: 'Eventuelle passerelle entre for' | Ligne {idx+2}]",
            "Code_Parcours_Cible": r.get('Code_Parcours_Cible'),
            "Type_Passerelle": r.get('Type_Passerelle'),
            "Justification": r.get('Justification')
        })

    return {"debouches": debouches, "passerelles": passerelles}

print("✅ 4 Outils agents métiers initialisés.")

✅ 4 Outils agents métiers initialisés.


## 6. Configuration LLM & Agent RAG avec Citations Vérifiables

In [34]:
from huggingface_hub import InferenceClient
from google.colab import userdata

# Récupération du HF_TOKEN enregistré dans Colab Secrets
HF_TOKEN = userdata.get("HFY_TOKEN")
client = InferenceClient(api_key=HF_TOKEN)

def call_llm(prompt):
    response = client.chat.completions.create(
        model="openai/gpt-oss-120b",
        messages=[{"role": "user", "content": prompt}],
        max_tokens=750,
        temperature=0.2
    )
    return response.choices[0].message.content

def ask_rag_agent(question, code_parcours_cible=None):
    """
    Orchestre la recherche hybride RAG, l'invocation des outils et la génération
    de réponses structurées avec des citations obligatoires.
    """
    # 1. Recherche Hybride RAG
    rag_results = recherche_hybride(question, top_k=4)

    context_blocks = []
    citations_list = []
    for res in rag_results:
        c = res['chunk']
        context_blocks.append(f"{c['citation']} {c['text']}")
        citations_list.append(c['citation'])

    # 2. Exécution conditionnelle d'outil métier si un parcours est spécifié
    outil_data = ""
    if code_parcours_cible:
        deb_pas = identifier_debouches_et_passerelles(code_parcours_cible)
        outil_data = f"\n[DONNÉES OUTIL DÉBOUCHÉS/PASSERELLES ({code_parcours_cible})]: {json.dumps(deb_pas, ensure_ascii=False)}"

    contexte_total = "\n---\n".join(context_blocks) + outil_data


    # 3. Toutes les filières
    q_lower = question.lower()

    # Si l'utilisateur demande une liste globale/exhaustive
    if any(m in q_lower for m in ["toutes les filières", "tous les filières", "tout les filières", "tous les parcours", "tout les parcours", "toutes les parcours", "toutes les formations", "liste des filières", "liste des formations", "liste des parcours"]):
        donnees_completes = lister_toutes_les_filieres()
        contexte_total = f"[DONNÉES EXHAUSTIVES DES FILIÈRES]: {json.dumps(donnees_completes, ensure_ascii=False)}"
    else:
        # Recherche RAG classique pour les questions spécifiques
        rag_results = recherche_hybride(question, top_k=5)
        context_blocks = [f"{c['chunk']['citation']} {c['chunk']['text']}" for c in rag_results]
        contexte_total = "\n---\n".join(context_blocks)

    # 4. Prompt structuré imposant la citation de source
    prompt = f"""Tu es ORIENT'IA, un assistant expert en orientation académique et professionnelle.

Consignes de rédaction :
1. Rédige une réponse claire, synthétique, fluide et directement compréhensible.
2. Structure la réponse avec soin (introduction directe, points clés à puces, synthèses et débouchés si pertinent).
3. Traçabilité des sources (VRAIES VALEURS DE SOURCE) :
   - N'affiche PAS de balises techniques brutes de type `[Feuille: '...' | Ligne X]`.
   - À la place, mentionne explicitement la VRAIE SOURCE des données de manière naturelle et lisible (ex: `(Source : Passerelles entre formations)`, `(Source : Programme & Matières IGGLIA)`, `(Source : Référentiel des compétences)`).
   - Indique la source réelle associée directement après l'information concernée.
   - À la fin de ton explication, ajoute une section " Sources des données" récapitulant clairement les vraies sources d'information consultées pour cette réponse.
4. Base-toi exclusivement sur les données collectées ci-dessous sans halluciner d'informations.

CONTEXTE ET DONNÉES COLLECTÉES :
{contexte_total}

QUESTION DE L'UTILISATEUR :
{question}

RÉPONSE CLAIRE ET PRÉCISE :"""
    return call_llm(prompt)

## 7. Démonstration des Outils & Tests du RAG

In [35]:
# Test 1: Utilisation directe des outils agents métiers
print("=== TEST OUTIL 1 : Rechercher Formation (IGGLIA Niveau 1) ===")
print(rechercher_formation(code_parcours="IGGLIA", niveau=1))

print("\n=== TEST OUTIL 3 : Calcul Score d'Adéquation ===")
profil = ["programmation", "bases de données", "intelligence artificielle", "statistiques"]
print(calculer_score_adequation(profil, "IGGLIA"))

print("\n=== TEST OUTIL 4 : Débouchés & Passerelles ===")
print(identifier_debouches_et_passerelles("IGGLIA"))

=== TEST OUTIL 1 : Rechercher Formation (IGGLIA Niveau 1) ===
[{'citation': "[Feuille: '01_Formations_Matieres' | Ligne 2]", 'Code_Parcours': 'IGGLIA', 'Nom_Parcours': 'IGGLIA', 'Niveau': 1, 'Diplôme': 'Bacc +1', 'Matières': 'Algèbre, Algorithmes, Analyse, Bases de données, Comptabilité, Français, HTML/CSS, Informatique scientifique, Mathématique Discrètes, Mathématique Financière, Organisation, PASCAL, Probabilités- Statistiques, Structures de données', 'Description': "Toutes les entreprises (publiques ou privées) ne peuvent plus se passer de l'outil informatique surtout l'informatique appliquée à la gestion. La filière Informatique de Gestion Génie Logiciel et Intelligence Artificielle est une filière dont l'objectif est la formation d'Ingénieurs et de Techniciens Supérieurs capables de maîtriser toutes les techniques informatiques relatives à la gestion des entreprises."}]

=== TEST OUTIL 3 : Calcul Score d'Adéquation ===
{'citation': "[Feuille: 'Compétences développées' | Ligne 2]"

In [55]:
# Test 2: Pose de question complète au système Agent RAG avec citations
question = "Quelles sont les filières proposées pour les personnes ayant un BAC Technique."
reponse = ask_rag_agent(question, code_parcours_cible="")

print("\n--- RÉPONSE DU RAG AVANCÉ D'ORIENTATION ---")
print(reponse)


--- RÉPONSE DU RAG AVANCÉ D'ORIENTATION ---
**Filières accessibles aux titulaires d’un BAC Technique à l’ISPM**

| Parcours | Mention | Niveau (diplôme) | Objectif principal | Principales matières enseignées | Débouchés typiques |
|----------|---------|------------------|--------------------|--------------------------------|--------------------|
| **ISAIA** | Informatique et Télécommunication | Bacc + 1 (Licence 1) | Appliquer les méthodes statistiques et informatiques aux domaines économiques | Algèbre, Algorithmes, Analyse, Bases de données, Combinatoire & Probabilités, HTML/CSS, Informatique scientifique, Macro‑/Micro‑économie, Statistique appliquée, etc. | Banques, entreprises industrielles, entreprises commerciales (secteur économique) (Source : 01_Formations_Matieres – INF‑ISAIA) |
| **GCA** | Génie Industriel et Génie Civil | Bacc + 1 (Licence 1) | Former des ingénieurs capables de concevoir, réaliser et gérer des infrastructures urbaines et rurales | Algèbre, Analyse, Dessin, 